# 38 · Skill 仓库 scaffolding + 团队分发

> **学习目标**：把「4 个 Skill 散装」升级成「**可版本化、可分发的 Skill 仓库**」—— 单个 git 仓、`README` 索引、`plugin` 系统、按角色（user / project / team）分发的 3 种加载路径。
>
> **预备**：33-37 跑过。
>
> **为什么重要**：当你有 8+ 个 Skill 时，**文件系统结构比单个 Skill 重要**。好的仓库 scaffolding = 团队新人 5 分钟上手、CI 自动校验、可一键安装。

In [ ]:
import os, shutil, json, subprocess
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable

SBX = Path('./_skill_sandbox').resolve()
if SBX.exists(): shutil.rmtree(SBX)
SBX.mkdir()
print('沙箱就绪')

## 1. 推荐仓库结构（个人/小团队）

```
skills/                            # 仓库根
├── README.md                      # 索引：所有 Skill 一览
├── CHANGELOG.md                  # 改了哪个 Skill 的 description / Steps
├── install.sh                    # 一键装到 ~/.claude/skills/（可选）
└── skills/
    ├── commit-pr/                 # 单个 Skill
    │   ├── SKILL.md
    │   ├── pr_template.md
    │   └── CHANGELOG.md
    ├── audit-rag/
    │   └── SKILL.md
    ├── study-topic/
    │   ├── SKILL.md
    │   └── NOTES_TEMPLATE.md
    └── ...
```

**关键设计**：**每个 Skill 是独立目录**（可单独 git mv / 复制到其他仓）。
**`CHANGELOG.md` 记 description / Steps 变更**（最容易被遗忘的元信息）。

In [ ]:
def write_file(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

REPO = SBX / 'my-skill-repo'
SKILLS_DIR = REPO / 'skills'
write_file(REPO / 'README.md', '''# My Skill Repo

> 一组 Claude Code Skill 集合。

## 全部 Skill

| Skill | 用途 | 版本 |
|-------|------|------|
| [commit-pr](skills/commit-pr/) | 提交 + 推 PR | 0.1.0 |
| [audit-rag](skills/audit-rag/) | 审计 rag_project 配置 | 0.1.0 |
| [study-topic](skills/study-topic/) | 深入研究 + 自动存笔记 | 0.1.0 |

## 装
```bash
./install.sh    # 复制到 ~/.claude/skills/
```

## 维护约定

- 每个 Skill 一个子目录
- `description` 改了必须改 `CHANGELOG.md`
- 每周跑一遍 `tests/recall_test.py`（如果存在）
''')
print('已建仓库 README')

In [ ]:
import yaml

def make_skill_in_repo(name, description, body, version='0.1.0'):
    d = SKILLS_DIR / name
    d.mkdir(parents=True, exist_ok=True)
    fm = {'name': name, 'description': description, 'version': version}
    front = '---' + chr(10) + yaml.safe_dump(fm, allow_unicode=True, sort_keys=False).rstrip() + chr(10) + '---'
    (d / 'SKILL.md').write_text(front + chr(10) + chr(10) + body, encoding='utf-8')
    today = datetime.now().strftime('%Y-%m-%d')
    cl = '# Changelog' + chr(10) + chr(10) + '## [' + version + '] ' + today + chr(10) + '- 初次创建' + chr(10)
    (d / 'CHANGELOG.md').write_text(cl, encoding='utf-8')
    return d

make_skill_in_repo('commit-pr', 'commit + open PR', '# Steps' + chr(10) + '1. git status + diff' + chr(10) + '2. craft commit + PR' + chr(10) + '3. push + gh pr create')
make_skill_in_repo('audit-rag', 'audit RAG config + recent changes', '# Steps' + chr(10) + '1. spawn sub-agent to read config + git log + eval set' + chr(10) + '2. return 150-word summary + 3 issues')
make_skill_in_repo('study-topic', 'deep-dive study of a topic', '# Steps' + chr(10) + '1. /search-web' + chr(10) + '2. /summarize' + chr(10) + '3. /write-notes')
print('created 3 skills')


## 2. install.sh —— 一键装到目标位置

**两种装入路径**（install.sh 支持参数选择）：
1. **用户级**：`~/.claude/skills/` —— 个人偏好
2. **项目级**：`<project>/.claude/skills/` —— 团队规范（应 git commit）

In [ ]:
write_file(REPO / 'install.sh', '''#!/usr/bin/env bash
# 一键装 skill 到指定位置
# 用法: ./install.sh [user|project] [path]
set -e
TARGET="${1:-user}"
REPO_SKILLS="$(dirname "$(realpath "$0")")/skills"
if [ "$TARGET" = "user" ]; then
    DEST="${HOME}/.claude/skills"
elif [ "$TARGET" = "project" ]; then
    DEST="${2:-./.claude/skills}"
else
    echo "usage: $0 [user|project] [path]"
    exit 1
fi
echo "→ 装到 $DEST"
mkdir -p "$DEST"
for skill_dir in "$REPO_SKILLS"/*/; do
    name="$(basename "$skill_dir")"
    if [ -d "$DEST/$name" ]; then
        echo "  [skip] $name 已在"; continue
    fi
    cp -r "$skill_dir" "$DEST/$name"
    echo "  [ok]   $name"
done
echo "完成。"
''')
(REPO / 'install.sh').chmod(0o755)
print(f'已写 install.sh ({(REPO / "install.sh").stat().st_size} bytes)')

In [ ]:
# 演示装到项目级 + 验证
project_root = SBX / 'demo-project'
project_root.mkdir()
proj_skills = project_root / '.claude' / 'skills'
proj_skills.mkdir(parents=True)

import subprocess as sp
result = sp.run(
    ['bash', str(REPO / 'install.sh'), 'project', str(proj_skills)],
    capture_output=True, text=True, encoding='utf-8'
)
print(result.stdout)
print(f'\n项目级 .claude/skills/ 内容:')
for p in sorted(proj_skills.iterdir()):
    print(f'  {p.name}/')
    for q in p.iterdir():
        print(f'    {q.name}')

## 3. 3 种加载路径（user / project / team）

**优先级**：project > user > default（后定义覆盖前）

In [ ]:
# 演示「项目级 Skill 覆盖 user 级」
user_skills_dir = SBX / 'demo-home' / '.claude' / 'skills'
user_skills_dir.mkdir(parents=True)

shutil.copytree(REPO / 'skills' / 'commit-pr', user_skills_dir / 'commit-pr')
print('用户级装入 commit-pr')

# 项目级覆盖（同名 Skill 走团队专版）
proj_override = proj_skills / 'commit-pr'
write_file(proj_override / 'SKILL.md', '''---
name: commit-pr
description: [PROJECT] Always include `Refs: #issue` in PR title. Trigger: "commit", "create PR", "提 PR".
---

# Steps
1. git status + diff
2. **always include `Refs: #N` if issue number mentioned**
3. push + gh pr create
''')
print('项目级覆盖 commit-pr（强制加 Refs 标签）')

In [ ]:
# 实现 SkillLoader 多级加载（project > user > default）
def load_with_priority(roots: list[Path]) -> dict[str, dict]:
    """later in roots = higher priority (覆盖)."""
    skills = {}
    for root in roots:
        if not root.is_dir():
            continue
        for entry in sorted(root.iterdir()):
            if not entry.is_dir():
                continue
            skill_md = entry / 'SKILL.md'
            if not skill_md.is_file():
                continue
            fm = yaml.safe_load('\n'.join(skill_md.read_text(encoding='utf-8').splitlines()[1:-1]))
            skills[fm['name']] = {
                'description': fm.get('description', ''),
                'from': str(root),
            }
    return skills

all_roots = [user_skills_dir, proj_skills]
loaded = load_with_priority(all_roots)
print(f'加载顺序: {all_roots}')
print(f'\n加载到的 Skill（含来源）:')
for n, info in loaded.items():
    print(f'  {n}')
    print(f'    desc: {info["description"][:80]}')
    print(f'    from: {info["from"]}')

## 4. CHANGELOG 自动维护

**核心痛点**：description 改了 = 召回行为变了，但**没人记得改 CHANGELOG**。

**修法**：写一个 `git diff` 后钩子，**自动检查 SKILL.md 改动 + 提示更新 CHANGELOG**。

In [ ]:
# 写个 git-pre-commit 钩子，提醒「你改了 SKILL.md 但没改 CHANGELOG」
write_file(REPO / 'git-hooks' / 'pre-commit', '''#!/usr/bin/env bash
# 每次 commit 自动检查：SKILL.md 改了但 CHANGELOG.md 没改？
set -e
STAGED=$(git diff --cached --name-only)
for f in $STAGED; do
    case "$f" in
        skills/*/SKILL.md)
            dir="$(dirname "$f")"
            if [ -f "$dir/CHANGELOG.md" ]; then
                if ! echo "$STAGED" | grep -q "^$dir/CHANGELOG.md"; then
                    echo "改了 $f 但没改 CHANGELOG.md"
                    echo "   自动追加一行到 $dir/CHANGELOG.md? [y/N]"
                    read -r ans
                    if [ "$ans" = "y" ]; then
                        date="$(date +%Y-%m-%d)"
                        echo "- $date: auto note — commit $f but CHANGELOG not edited" >> "$dir/CHANGELOG.md"
                        git add "$dir/CHANGELOG.md"
                        echo "已自动追加一行"
                    else
                        echo "不自动追加，commit 终止"
                        exit 1
                    fi
                fi
            fi
            ;;
    esac
done
''')
(REPO / 'git-hooks' / 'pre-commit').chmod(0o755)
print('已写 pre-commit hook 脚本')

# 模拟一次 commit 检查
init_repo = REPO / 'init-tmp'
init_repo.mkdir()
sp.run(['git', 'init', '-b', 'main'], cwd=init_repo, capture_output=True)
sp.run(['git', 'config', 'user.email', 'x@y'], cwd=init_repo, capture_output=True)
sp.run(['git', 'config', 'user.name', 't'],   cwd=init_repo, capture_output=True)
shutil.copy(REPO / 'install.sh', init_repo / 'install.sh')
sp.run(['git', 'add', 'install.sh'], cwd=init_repo, capture_output=True)
sp.run(['git', 'commit', '-m', 'init'], cwd=init_repo, capture_output=True, text=True, encoding='utf-8')
shutil.copy(REPO / 'skills' / 'commit-pr' / 'SKILL.md', init_repo / 'commit-pr.md')
sp.run(['git', 'add', 'commit-pr.md'], cwd=init_repo, capture_output=True)

import os
os.environ['GIT_AUTHOR_NAME'] = 't'
hook = init_repo / '.git' / 'hooks' / 'pre-commit'
shutil.copy(REPO / 'git-hooks' / 'pre-commit', hook)
hook.chmod(0o755)

result = sp.run(
    ['git', 'commit', '-m', 'change desc'],
    cwd=init_repo, capture_output=True, text=True, encoding='utf-8',
    input='N\n'  # 拒绝自动追加
)
print('exit code:', result.returncode)
print('stdout:', result.stdout)
print('stderr:', result.stderr)

In [ ]:
shutil.rmtree(init_repo, ignore_errors=True)
print('tmp cleanup')

## 5. plugin / marketplace —— 大团队级分发

**超 8 个 Skill 时**：考虑打成 **plugin**，让团队用「`/plugin install team-skills`」一键装。
**Anthropic 官方 plugin 系统**包括：
- `marketplace.json` 索引（多 plugin 元数据）
- plugin 内部仍是标准 Skill 结构
- 通过 marketplace URL 安装

我们写一个最小化的 `marketplace.json` 演示。

In [ ]:
write_file(REPO / 'marketplace.json', '''{
  "name": "my-team-skills",
  "owner": {
    "name": "Your Team",
    "email": "team@example.com"
  },
  "plugins": [
    {
      "name": "core",
      "description": "Core skills everyone needs",
      "source": "./skills",
      "skills": ["commit-pr", "audit-rag"]
    },
    {
      "name": "research",
      "description": "Research / study skills (optional)",
      "source": "./skills",
      "skills": ["study-topic"]
    }
  ]
}
''')
print('已建 marketplace.json')
print()
print('这是 Anthropic plugin 系统要的最小 schema:')
print('  - name: marketplace 名')
print('  - plugins[]: 每个 plugin 是一组 Skill 的"套餐"')
print('  - skills[]: 该 plugin 包含哪些 Skill 名字')
print()
print('→ 团队新人跑 `/plugin install my-team-skills` → 选 plugin `core` → 3 个 Skill 一次性装上。')

In [ ]:
# 验证仓库结构
print('最终仓库结构:')
for p in sorted(REPO.rglob('*')):
    rel = p.relative_to(REPO)
    depth = len(rel.parts) - 1
    indent = '  ' * depth
    icon = '目录' if p.is_dir() else '文件'
    print(f'{indent}{icon} {p.name}{"/" if p.is_dir() else ""}')

In [ ]:
shutil.rmtree(SBX, ignore_errors=True)
print('沙箱清理')

## 深入思考

1. **为什么每个 Skill 一个 CHANGELOG 不直接放仓根？**
   - 单个 Skill 跨仓迁移时**它的历史跟着走**。仓根 CHANGELOG 失去**粒度**。
2. **项目级 Skill 该不该 git commit？**
   - **必须 commit**。否则 git clone 项目后新人没这些 Skill。**约定 > 文档**。
3. **plugin vs 多仓 怎么选？**
   - 单团队 1 仓 plugin 够；多团队 / 多语言 / 公私分离 → 多仓 + marketplace 索引。
4. **能否用 `chezmoi` / `stow` 同步 settings？**
   - 很多人这么做。把 `~/.claude/settings.json` + `~/.claude/skills/` 全链到 dotfiles 仓。**多机一致的最佳实践**。
5. **怎么知道哪些 Skill 该砍？**
   - 跑 35 号的 `RecallTester`，**3 个月不被召的 Skill 即可砍**。description 写得再烂，**3 个月没有触发** = 真没人需要。

**改一改**：
- 给你当前的 Skill 库加一份自动生成的 README（扫所有 Skill 的 description 凑成表格）
- 加 `tests/recall_test.py`，让 CI 在 PR 时跑回归（如果 description 改了 → 报告召回率掉没掉）

## 自检 ✅

- [ ] 默写仓库根结构（README / CHANGELOG / install.sh / skills/）
- [ ] 解释「为什么每个 Skill 一个 CHANGELOG」
- [ ] 解释「项目级 Skill 必须 git commit」的原因
- [ ] 解释「plugin vs 多仓」何时用哪个
- [ ] 默写 pre-commit hook 检查 SKILL.md 改动的逻辑

## 03-Claude-Skills 全部完成

**走完 33-38 你应该具备**：
- 在沙箱里建出 `~/.claude/skills/` 镜像，理解协议
- 用 5 段模板写出 production-grade Skill（含反例 + checklist）
- 写 `RecallTester` 跑 20+ query 回归测试
- 组合 Skill → sub-agent + meta-Skill 工作流
- 装 Skill 时自动配 permissions + hooks
- 建可分发、可版本化的 Skill 仓库

**下一步**：→ [04-微调](../../../04-模型微调-Finetuning/) 或 [07-Capstone 项目 2](../../../07-综合项目-Capstone/)